# 样本方差的抽样分布（χ²分布）

> 对应代码：`MathStatsCode/code_in_notes/Chap.6/standard_normal_var.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：通过20000次模拟验证：正态总体下 (n-1)S²/σ² 服从 χ²(n-1)，并与 χ²(9) 密度曲线对比。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.6`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.6")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

本 notebook 通过模拟验证**样本方差的抽样分布**。程序 `std_normal_var` 用 `syntax` 接受可选参数 `obs/mu/sigma`（默认 n=10、μ=0、σ=1），生成 n 个正态样本后返回 `std_var=(n-1)·S²/σ²`——注意 `r(Var)` 是 Stata 的**无偏样本方差**（分母 n-1），因此该统计量在理论恰好服从 χ²(n-1)。
随后 `simulate std_var=r(std_var), reps(20000)` 把程序重复20000次生成统计量的经验分布；`sort` 排序后与 `chi2den(9,·)`（自由度 n-1=9 的 χ² 密度）叠加成 twoway 图：直方图是模拟分布、曲线是理论密度，两者几乎重合即验证了结论。这是说明“为什么用 (n-1)S²/σ² 而不是 n 倍”的经典模拟。


In [ ]:
%%stata
cap program drop std_normal_var
program define std_normal_var, rclass
	version 12
	syntax [,obs(integer 10) mu(real 0) sigma(real 1)]
	drop _all
	set obs `obs'
	tempvar x
	gen `x'=`sigma'*rnormal()+`mu'
	su `x'
	return scalar std_var=(`obs'-1)*r(Var)/(`sigma'^2)
end

simulate std_var=r(std_var), reps(20000): std_normal_var

sort std_var
gen chi2_den=chi2den(9,std_var)
label variable chi2_den "Chi2 Density"
twoway (hist std_var, density) (line chi2_den std_var), graphr(fcolor(white) color(white)) xtitle("")
